In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.nn import GCNConv, GATConv, GINConv

# ==============================================================================
# 1. MODEL ARCHITECTURES
# ==============================================================================

class GCNNet(nn.Module):
    """GCN (Graph Convolutional Network) with symmetric degree normalization."""
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x


class GATNet(nn.Module):
    """GAT (Graph Attention Network) with multi-head self-attention."""
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int, heads: int = 8):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = F.dropout(x, p=0.6, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return x


class GINNet(nn.Module):
    """GIN (Graph Isomorphism Network) with injective sum + MLP (Multi-Layer Perceptron)."""
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int):
        super().__init__()
        mlp1 = nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels)
        )
        self.conv1 = GINConv(mlp1, train_eps=True)

        mlp2 = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, out_channels)
        )
        self.conv2 = GINConv(mlp2, train_eps=True)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x


# ==============================================================================
# 2. TRAINING & EVALUATION HARNESS
# ==============================================================================

def train_epoch(model: nn.Module, data, optimizer: torch.optim.Optimizer, criterion: nn.Module) -> float:
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate(model: nn.Module, data) -> tuple:
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=-1)

    train_acc = (pred[data.train_mask] == data.y[data.train_mask]).sum().item() / data.train_mask.sum().item()
    val_acc = (pred[data.val_mask] == data.y[data.val_mask]).sum().item() / data.val_mask.sum().item()
    test_acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()

    return train_acc, val_acc, test_acc


def run_benchmark(model_name: str, model: nn.Module, data, epochs: int = 150, lr: float = 0.01, weight_decay: float = 5e-4):
    device = data.x.device
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_test_acc = 0.0
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        loss = train_epoch(model, data, optimizer, criterion)
        train_acc, val_acc, test_acc = evaluate(model, data)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_test_acc = test_acc

    total_time = time.time() - start_time
    time_per_epoch_ms = (total_time / epochs) * 1000

    return {
        "Model": model_name,
        "Best Val Acc": f"{best_val_acc * 100:.2f}%",
        "Test Acc": f"{best_test_acc * 100:.2f}%",
        "Total Time (s)": f"{total_time:.2f}",
        "Time/Epoch (ms)": f"{time_per_epoch_ms:.2f}"
    }


# ==============================================================================
# 3. BENCHMARK EXECUTION (CORA DATASET)
# ==============================================================================

if __name__ == "__main__":
    torch.manual_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running Benchmark on Device: {device}\n")

    # Load Cora Benchmark Dataset
    dataset = Planetoid(root="data/Planetoid", name="Cora", transform=T.NormalizeFeatures())
    data = dataset[0].to(device)

    num_features = dataset.num_features
    num_classes = dataset.num_classes
    hidden_dim = 16

    models = {
        "GCN": GCNNet(in_channels=num_features, hidden_channels=hidden_dim, out_channels=num_classes),
        "GAT": GATNet(in_channels=num_features, hidden_channels=8, out_channels=num_classes, heads=8),
        "GIN": GINNet(in_channels=num_features, hidden_channels=hidden_dim, out_channels=num_classes)
    }

    results = []
    for name, model in models.items():
        print(f"Training {name} (Graph Neural Network)...")
        res = run_benchmark(name, model, data, epochs=150, lr=0.01, weight_decay=5e-4)
        results.append(res)

    # Display Benchmark Results
    print("\n" + "=" * 65)
    print(f"{'Model':<10} | {'Best Val Acc':<14} | {'Test Acc':<12} | {'Time/Epoch (ms)':<15}")
    print("-" * 65)
    for r in results:
        print(f"{r['Model']:<10} | {r['Best Val Acc']:<14} | {r['Test Acc']:<12} | {r['Time/Epoch (ms)']:<15}")
    print("=" * 65)